# Multi-Agent LLM Discussion for Paper Reviewing

**Workshop — 30 to 60 minutes**

In this workshop you will build a multi-agent peer-review system. By the end you will have a working Python script that simulates structured academic peer review through multiple LLM agents - and you will have seen whether and why that produces better critiques than a single prompt.

Loading and prompting models is not included in the scope of this workshop. We will instead be using a specialized library ([SynDisco](https://github.com/dimits-ts/syndisco)), which abstracts away loading, prompting, and organizing LLM agents. If you want to explore the concepts introduced in this lab, you could consult the [online documentation](https://dimits-ts.github.io/syndisco), although this will not be necessary for completion.

### Objectives

By the end of this session you will be able to:

1. **Design** a role-based multi-agent discussion system to solve a specific problem
2. **Implement** a multi-round interaction loop using Python
3. **Compare** single-prompt vs multi-agent approaches
4. **Justify** for what tasks synthetic discussion is appropriate

---

### Prerequisites

- Basic Python knowledge
- Basic familiarity with LLMs

---

### License

This notebook is licensed under the GPLv3 license.
```
This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU General Public License for more details.

You should have received a copy of the GNU General Public License
along with this program.  If not, see <http://www.gnu.org/licenses/>.

You may contact the author at dim.tsirmpas@aueb.gr
```
---


## Setup

In [1]:
%set_env CUDA_VISIBLE_DEVICES=2

env: CUDA_VISIBLE_DEVICES=2


%pip install syndisco --quiet

In [ ]:
import syndisco
from syndisco import TransformersModel


# You may want to change this depending on your hardware and needs.
# We recommend picking models from huggingface: https://huggingface.co
# 
model = TransformersModel(
    model_path="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    name="base_model",
    max_out_tokens=1500,
)
syndisco.logging_setup(
    print_to_terminal=True,
    write_to_file=False,
    level="warning",
    use_colors=True
)


## Reviewing a paper

Before we get into multi-agent systems, let’s start simple.

Below is a short excerpt from a research paper. Your task is to review it using a language model. You can prompt the model however you like-there are no constraints yet. Treat this as an initial experiment.

In [3]:
# --- Paper abstract to review -----------------------------------------------
ABSTRACT = """
Title: Attention Is All You Need

Abstract:
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and
convolutions entirely. Experiments on two machine translation tasks show these
models to be superior in quality while being more parallelizable and requiring
significantly less time to train.
"""

Prompting is fairly simple. We just need to define a system prompt (to clue our model in what the task is), and a user prompt (which contains the actual "meat" of our request).

In [4]:
BASELINE_SYSTEM = "You are an expert academic reviewer. Review the paper excerpt provided."
BASELINE_USER   = f"Please review the following paper abstract:\n\n{ABSTRACT}"

baseline_review = model.prompt(BASELINE_SYSTEM, BASELINE_USER)

print("=" * 60)
print("BASELINE — single-prompt review")
print("=" * 60)
print(baseline_review)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BASELINE — single-prompt review
### Review of the Paper Abstract

**Title:** "Attention Is All You Need"

**Abstract:**
"The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train."

#### Strengths:

1. **Clear Introduction:**
   - The abstract effectively introduces the current state of the art in sequence transduction models, highlighting the use of recurrent and convolutional neural networks (RNNs/CNNs) and the role of attention mechanisms.
   
2. **Proposed Contribution:**
   - It clear

There is of course no automated way of evaluating this review. What we can do is have a look at the output ourselves, and determine some task-specific quality checks. Since our task is academic reviewing these checks could look something like this:

---

* [ ] **On track?**
  Does the response actually address the question?

* [ ] **Clear + readable?**
  Is it easy to follow, or a bit messy?

* [ ] **Some real thinking?**
  Does it go beyond obvious or generic points?

* [ ] **Balanced?**
  Does it consider more than one angle, or just present a single take?

* [ ] **Any pushback?**
  Does it question assumptions or just accept them?

* [ ] **Feels complete?**
  Do you feel like you got a useful answer, or something partial/shallow?

* [ ] **Feels realistic?**
  Does this read like an actual human-generated review? Could you discern between the two?

* [ ] **Overall feel**
  Does this feel genuinely helpful for your task?

* [ ] **Most importantly**
  Did you actually get any insights on the paper?
  
*(We'll come back to this exact checklist later)*

---

## Adding spice to the reviews

A simple way of generating more diverse and targeted answers is by adding more details to the system prompt. While this can be trivially achieved by directly changing the system prompt above (try it out!), we can use a standard template to make our life easier when we define multiple LLM paricipants.

Our library already takes care of basic templating using the `Actor` class. Let's define a very skeptical reviewer using the following attributes:
- **`persona`** — a dict of attributes that shape its voice (role, expertise, style)
- **`context`** — what the discussion is about (ideally shared across all agents)
- **`instructions`** — same as before, but can be customized for each agent

In [5]:
from syndisco import Actor


REVIEW_CONTEXT = (
    f"You are participating in a structured academic peer-review discussion. "
    f"The paper under review is:\n\n{ABSTRACT}"
)

REVIEW_INSTRUCTIONS = "Write a full, professional review of the article for a prestigious journal."

skeptical_reviewer = Actor(
    model=model,
    persona={
        "role": "Skeptical Reviewer",
        "expertise": "critical analysis, assumption-testing, identifying gaps",
        "style": "challenging, rigorous, devil's advocate",
    },
    context=REVIEW_CONTEXT,
    instructions=REVIEW_INSTRUCTIONS,
    name="SkepticalReviewer",
)

skeptical_review = skeptical_reviewer.speak()
print("=" * 60)
print("SKEPTICAL — single-prompt review")
print("=" * 60)
print(skeptical_review)

SKEPTICAL — single-prompt review
**Review by SkepticalReviewer**

**Title: Attention Is All You Need**

**Introduction:**
The paper "Attention Is All You Need" proposes a novel architecture for sequence transduction tasks, specifically machine translation, using only attention mechanisms. This work challenges the traditional reliance on recurrent neural networks (RNNs) and convolutional neural networks (CNNs), suggesting that attention alone can achieve state-of-the-art performance. While the claims are intriguing, several critical aspects need thorough examination.

**Methodology:**
The authors present a transformer model that relies solely on self-attention mechanisms. They argue that this approach is more efficient and scalable compared to existing architectures. However, the methodology section could benefit from more detailed explanations of how the attention mechanism is implemented and how it differs from previous approaches. Additionally, the paper should provide a clear compar

What have we gained by creating a different persona? Generally speaking, we have a whole other Point of View, which we would have otherwised missed. This is the core motivation of using different agents.



## Multi-agent prompting

Let's spice things up even further. Why not simulate the entire first round of the review?

A normal peer-review takes anywhere from two to four reviewers. We already have a skeptical reviewer, so think about what other *points of view* would be beneficial for the review.

We should also add a **meta-reviewer** as a participant here — they should join at the end of the review cycle, when all other reviews have been posted.

In [6]:
# --- Statistical Reviewer ---------------------------------------------------
statistical_reviewer = Actor(
    model=model,
    persona={
        "role": "Statistical Reviewer",
        "expertise": "statistics, experimental design, causal inference, reproducibility",
        "style": "rigorous, detail-oriented, critical but constructive",
    },
    context=REVIEW_CONTEXT,
    instructions=REVIEW_INSTRUCTIONS,
    name="StatisticalReviewer",
)

# --- NLP Expert Reviewer ----------------------------------------------------
nlp_reviewer = Actor(
    model=model,
    persona={
        "role": "NLP Expert Reviewer",
        "expertise": "natural language processing, deep learning, LLMs, benchmarking",
        "style": "scholarly, literature-aware, technically grounded",
    },
    context=REVIEW_CONTEXT,
    instructions=REVIEW_INSTRUCTIONS,
    name="NLPReviewer",
)

# --- Author -----------------------------------------------------------------
author = Actor(
    model=model,
    persona={
        "role": "Author",
        "expertise": "the submitted paper and its contributions",
        "style": "defensive but professional, transparent, evidence-based",
    },
    context=REVIEW_CONTEXT,
    instructions=(
        "You are the author of the submitted paper. Respond to reviewer comments by:\n"
        "  • Addressing each critique directly and respectfully\n"
        "  • Providing clarifications, additional justification, or acknowledging limitations\n"
        "  • Proposing concrete revisions where appropriate\n"
        "  • Avoiding defensiveness; aim to improve the paper\n"
    ),
    name="Authors",
)

# --- Meta-Reviewer ----------------------------------------------------------
meta_reviewer = Actor(
    model=model,
    persona={
        "role": "Meta-Reviewer",
        "expertise": "synthesis, editorial judgment, final recommendations",
        "style": "balanced, decisive, structured",
    },
    context=REVIEW_CONTEXT,
    instructions=(
        "You have read all previous reviewer comments. Synthesise the discussion "
        "into a final structured judgment. Your output must contain:\n"
        "  • Summary of main strengths\n"
        "  • Summary of main weaknesses\n"
        "  • Overall recommendation (Accept / Major Revision / Reject) with a one-sentence rationale."
    ),
    name="MetaReviewer",
)

print("Agents created:")
for agent in [
    statistical_reviewer,
    nlp_reviewer,
    skeptical_reviewer,
    author,
    meta_reviewer,
]:
    print(f"\t* {agent.get_actor_name()}")

Agents created:
	* StatisticalReviewer
	* NLPReviewer
	* SkepticalReviewer
	* Authors
	* MetaReviewer


In round 1 each of the three reviewers should read the abstract with no prior discussion context.

In [ ]:
print("Generating round 1 independent reviews...\n")

round1_reviewers = [
    statistical_reviewer,
    nlp_reviewer,
    skeptical_reviewer,
]

# Each reviewer speaks independently — no history yet
print("Round 1 starting...\n")

for reviewer in round1_reviewers:
    print(f"{'─' * 60}")
    print(f"[{reviewer.get_actor_name()}]")
    
    opinion = reviewer.speak()  # runs + returns
    print(opinion)

print("─" * 60)
print("Round 1 complete.")

Generating round 1 independent reviews...

────────────────────────────────────────────────────────────
[StatisticalReviewer]
**Review of "Attention Is All You Need"**

**Submitted by:** User StatisticalReviewer

**Role:** Statistical Reviewer

**Expertise:** Statistics, Experimental Design, Causal Inference, Reproducibility

**Style:** Rigorous, Detail-Oriented, Critical but Constructive

---

**Introduction:**
The paper "Attention Is All You Need" proposes a novel approach to sequence transduction using a transformer model that relies solely on attention mechanisms. This work challenges the traditional reliance on recurrent neural networks (RNNs) and convolutional neural networks (CNNs) by presenting a simpler and potentially more efficient architecture.

**Strengths:**
1. **Innovative Architecture:** The authors introduce a transformer model that eliminates the need for recurrence and convolutions, focusing purely on attention mechanisms. This is a significant departure from existin

Our pipeline so far should encourage the presence of multiple POVs. Let's revisit the checklist from before, and determine whether utilizing multiple agents made a difference.

---

* [ ] **On track?**
  Does the response actually address the question?

* [ ] **Clear + readable?**
  Is it easy to follow, or a bit messy?

* [ ] **Some real thinking?**
  Does it go beyond obvious or generic points?

* [ ] **Balanced?**
  Does it consider more than one angle, or just present a single take?

* [ ] **Any pushback?**
  Does it question assumptions or just accept them?

* [ ] **Feels complete?**
  Do you feel like you got a useful answer, or something partial/shallow?

* [ ] **Feels realistic?**
  Does this read like an actual human-generated review? Could you discern between the two?

* [ ] **Overall feel**
  Does this feel genuinely helpful for your task?

* [ ] **Most importantly**
  Did you actually get any insights on the paper?
  
We should also add another very important question now:

* [ ] **Was it worth the trouble**?
  Using multiple agents meant spending some time designing them and waiting for their responses. Were the results you got worth the cost in time (and perhaps resources)? Why?

---

## Round 2 and Round 3 — Discussion and meta-review

You have perhaps identified a limitation in our approach so far; the agents respond in isolation, and can not take into account what the other agents have highlighted. This limitation is important for two reasons:

1. The review process after this point will not be **realistic**. In reality, after the first round of reviewing, the authors engage with the reviewers in order to respond to their questions. 
    - One of the reasons for building this reviewing pipeline is to simulate what could (not *would*!) happen in the real world. In this case, our setup is a special case of Agent-Based Modeling (ABM). ABM concerns itself with building simulated agents with simple rules, then allowing them to interact with each other to simulate complex human behavior. In our case, since our agents don't need explicit rules, but rather instructions, we are talking abouut Generative Agent-Based Modeling (GABM).
    - A motivation for using synthetic discussions then is to simulate what could have happened in a peer-review process. If you want to explore this part of the discussion, try it out by *adding your own paper to the top of the notebook* and re-running it.

2. The review process will not be **informative**. We can already see that agents may focus on different aspects of the work. Why shouldn't a critical reviewer take into consideration points the strengths mentioned by other reviewers?
    - Going back-and-forth would allow agents to iteratively bring up, discard, and fight over specific points raised in the paper. This process is dynamic, and requires no input from us.
    - If you want to explore what *information* you can extract using synthetic discussions, an abstract may not be enough. Why don't we try loading a whole paper? This will raise some issues, since we are using a local model. How would you circumvent them?

---

We will create a discussion between the reviewers defined above. You could also add an author; an agent that will attempt to defend the paper. How would you defend your paper? Add 

In [8]:
from syndisco import Discussion, RoundRobin


all_agents = [
    statistical_reviewer,
    nlp_reviewer,
    skeptical_reviewer,
    author,
    meta_reviewer,
]

# RoundRobin cycles through actors in the order they are given.
turn_manager = RoundRobin(actors=all_agents) 

discussion = Discussion(
    next_turn_manager=turn_manager,
    users=all_agents,
    # In synthetic discussions, initial starting points (hardcoded messages)
    # are usually referred to as "seed opinions"
    # Round-1 reviews are injected as seeds since the reviewers should 
    # now know the results of the first round of reviewing
    seed_opinions=opinions,
    seed_opinion_usernames=agent_names,
    # Each agent sees the last 6 messages as context (enough for all seeds + some turns)
    # You may need to turn this down if you are running this notebook in a machine
    # with relatively low VRAM, or if you decide to load a large document.
    history_context_len=6,
    # 3 updated critiques (round 2) + 1 meta-review (round 3)
    conv_len=4,
)

print("Running peer-review rounds 2 and 3...")
discussion.begin(verbose=True)

Running peer-review rounds 2 and 3...


  0%|          | 0/4 [00:00<?, ?it/s]

User StatisticalReviewer posted:
**Review of "Attention Is All You Need"**  **Introduction:** The paper
"Attention Is All You Need" by Vaswani et al. (2017) introduces a
novel architecture for sequence transduction tasks, specifically
machine translation, that relies solely on attention mechanisms. This
work challenges the traditional reliance on recurrent neural networks
(RNNs) and convolutional neural networks (CNNs), proposing that
attention alone can achieve state-of-the-art performance. The authors
present a transformer model that eliminates the need for recurrence
and convolutions, focusing purely on attention mechanisms. This
approach is claimed to offer several advantages, including improved
parallelizability and reduced training time.  **Strengths:** 1.
**Innovative Architecture:** The transformer model presented in this
paper is a significant departure from existing models, offering a
simpler and potentially more efficient architecture. This innovation
could lead to new insig

## Was it worth it?

Let's print the baseline single-prompt review alongside the meta-reviewer's final judgment.

In [9]:
# The meta-reviewer is always the last entry in the log
meta_review_entry = discussion.get_logs()[-1]

divider = "=" * 60

print(divider)
print("BASELINE — single-prompt review")
print(divider)
print(baseline_review)

print()
print(divider)
print(f"MULTI-AGENT — {meta_review_entry['name']} final judgment")
print(divider)
print(meta_review_entry['text'])

BASELINE — single-prompt review
### Review of the Paper Abstract

**Title:** "Attention Is All You Need"

**Abstract:**
"The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train."

#### Strengths:

1. **Clear Introduction:**
   - The abstract effectively introduces the current state of the art in sequence transduction models, highlighting the use of recurrent and convolutional neural networks (RNNs/CNNs) and the role of attention mechanisms.
   
2. **Proposed Contribution:**
   - It clear

Is the meta-review sufficient? Should we also include the discussion itself in our analysis? If so, should we keep the entire discussion? Should we summarize it?

These are all practical questions which you will need to answer depending on the task, what you are looking for, whether you want to maximize information extraction or simulate many peer-review processes, etc.

Let's now refer to the checklist one last time. 

---

* [ ] **On track?**
  Does the response actually address the question?

* [ ] **Clear + readable?**
  Is it easy to follow, or a bit messy?

* [ ] **Some real thinking?**
  Does it go beyond obvious or generic points?

* [ ] **Balanced?**
  Does it consider more than one angle, or just present a single take?

* [ ] **Any pushback?**
  Does it question assumptions or just accept them?

* [ ] **Feels complete?**
  Do you feel like you got a useful answer, or something partial/shallow?

* [ ] **Feels realistic?**
  Does this read like an actual human-generated review? Could you discern between the two?

* [ ] **Overall feel**
  Does this feel genuinely helpful for your task?

* [ ] **Most importantly**
  Did you actually get any insights on the paper?
  
* [ ] **Was it worth the trouble**?
  Were the results you got worth the cost in time (and perhaps resources)? Why?

---

## Summary

In this workshop you:

1. Attempted to solve a problem through a traditional **single-prompt** approach
1. Used instructions and personas to build an alternative **role-specialised multi-agent** system
1. Modeled the specific problem in terms of **interactions** between multiple agents
1. Used **synthetic discussions** to manage context, combine information, and dynamically shift the focus of the agents around different parts of the paper

Having run the four different setups, you now should have an intuition on the effort, time and resources needed for each LLM-led approach. Given a new research problem, you should be able to determine which of these approaches is appropriate, and how you can transform it into an appopriate, agent-based approach.

---

## What next?

There are many practical issues you will encounter when you scale up this approach. Notably:

- How do I run randomize experiments to ensure my simulations have enough coverage? 
    - See the library's [Experiment](https://dimits-ts.github.io/syndisco/generated/syndisco.DiscussionExperiment.html) class for a starting point.
- Do I use proprietary, API-based models (such as OpenAI or Anthropic models), or local models?
    - This debate is described in-depth in [this paper](https://arxiv.org/abs/2503.16505).
- How do I feed the model with large, extensive documents without running out of credits / VRAM?
    - You may need to check techniques such as document splitting, document summarization, or memory modules.
- How can I know whether my simulations are realistic?
    - LLM simulations can not produce results which will be replicated with certainty with humans, [no matter how much engineering goes into them](https://sociologica.unibo.it/article/view/19576). 
    - However, building a basic evaluation suite for your experiments can be indispensable. We point you to a [comprehensive paper for GABM evaluation](https://ieeexplore.ieee.org/abstract/document/10985773) (as we discussed when we defined the purpose of synthetic discussions), and [the paper above](https://arxiv.org/abs/2503.16505), which describes which of these methods may be used for synthetic discussion tasks.
- Can I use these techniques for LLM annotation?
    - [Yes](https://dimits-ts.github.io/syndisco/generated/syndisco.AnnotationExperiment.html).
- How do I apply this logic to \<X problem\>?
    - No one can help you. Good luck :).


You can use this notebook as a starting point from which you can start exploring these questions!